# 04 – XGBoost

**Goal:** Can XGBoost beat the baselines from notebook 03?
- Regression standard: CV RMSE 0.721 (≈ $72k), MAE 0.529, R² 0.611
- Classification standard: CV F1 0.828

**XGBoost** = a big model built from many small models (trees). Each new tree tries to fix the errors the earlier trees left.
Trees can learn bends (e.g. location: Latitude AND Longitude together), not only straight lines.

**Data:** same as notebook 03. All 8 features, 80/20 split, random_state=42. `expensive` = above the train median.

In [1]:
import time
import pandas as pd
import xgboost
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_validate

print("xgboost:", xgboost.__version__)

housing_df = fetch_california_housing(as_frame=True).frame

feature_columns = housing_df.columns.drop("MedHouseVal")
X = housing_df[feature_columns]
y = housing_df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

median_price_train = y_train.median()
y_train_expensive = (y_train > median_price_train).astype(int)
y_test_expensive = (y_test > median_price_train).astype(int)

print("Train:", X_train.shape, " Test:", X_test.shape)
print(f"Median price (train): {median_price_train:.3f}")

xgboost: 3.4.1
Train: (16512, 8)  Test: (4128, 8)
Median price (train): 1.798


## XGBoost regression: default settings

No scaler needed: a tree only asks "is it bigger than X?", and that answer stays the same whether the numbers are big or small (like lining people up by height in cm or inches: same order).

Same scores as the baseline (5-fold CV on the train set). Standard to beat: **RMSE 0.721 (≈ $72k)**.

In [2]:
from xgboost import XGBRegressor

xgboost_default_regressor = XGBRegressor(random_state=42)

start_time = time.perf_counter()
xgboost_default_regressor.fit(X_train, y_train)
xgb_regression_train_time_seconds = time.perf_counter() - start_time

xgb_regression_cv_results = cross_validate(
    xgboost_default_regressor, X_train, y_train, cv=5,
    scoring={
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2",
    },
)

xgb_default_cv_rmse = -xgb_regression_cv_results["test_rmse"].mean()
xgb_default_cv_mae = -xgb_regression_cv_results["test_mae"].mean()
xgb_default_cv_r2 = xgb_regression_cv_results["test_r2"].mean()

print(f"Training time: {xgb_regression_train_time_seconds:.3f} seconds")
print(f"CV RMSE: {xgb_default_cv_rmse:.3f}  (≈ ${xgb_default_cv_rmse * 100_000:,.0f})")
print(f"CV MAE:  {xgb_default_cv_mae:.3f}  (≈ ${xgb_default_cv_mae * 100_000:,.0f})")
print(f"CV R²:   {xgb_default_cv_r2:.3f}")

Training time: 0.180 seconds
CV RMSE: 0.477  (≈ $47,687)
CV MAE:  0.316  (≈ $31,614)
CV R²:   0.830
